In [43]:
import wandb

api = wandb.Api()
run_paths = {}
run_paths['s_1ep'] = 'mol-llm/mol-llm/rs8jl0bv'
run_paths['s+g_1ep'] = 'mol-llm/mol-llm/j8e6d79c'
hist = {}
for k in run_paths:
    run = api.run(run_paths[k])
    # get training log
    hist[k] = run.history()

metrics = [
    'exact_match_ratio',
    'bleu_selfies',
    'RDK_FTS',
    'MACCS_FTS',
    'morgan_FTS',
    'validity_ratio',
    'mae',
    'rmse',
    'roc_auc',
    'bleu2',
    'bleu4',
    'rouge1',
    'rouge2',
    'rougeL',
    'meteor'
]

tasks = [
    'smol-property_prediction-esol',
    'smol-property_prediction-lipo',
    'qm9_homo',
    'qm9_lumo',
    'qm9_homo_lumo_gap',
    'bace',
    'smol-property_prediction-bbbp',
    'smol-property_prediction-clintox',
    'smol-property_prediction-hiv',
    'smol-property_prediction-sider',
    'forward_reaction_prediction',
    'smol-forward_synthesis',
    'retrosynthesis',
    'smol-retrosynthesis',
    'reagent_prediction',
    'chebi-20-text2mol',
    'smol-molecule_generation',
    'chebi-20-mol2text',
    'smol-molecule_captioning'
]

# generate all task/metric combinations (for validation metrics)
for task in tasks:
    for metric in metrics:
        taking_cols = [f"val/{task}/{metric}" for metric in metrics for task in tasks]
taking_cols.append("epoch")
taking_cols.append("trainer/global_step")
remove_cols = [
"val/smol-property_prediction-lipo/mae",
"val/smol-property_prediction-esol/mae",
"val/qm9_homo/rmse",
"val/qm9_lumo/rmse",
"val/qm9_homo_lumo_gap/rmse",
]

existing_taking_cols = [c for c in taking_cols]
existing_taking_cols = [c for c in existing_taking_cols if c not in remove_cols]
# keep only those actually present in history

existing_taking_cols = [col for col in existing_taking_cols if col in hist['s+g_1ep'].columns]
# sort tasks in existing_taking_cols  are aligneg with the order in tasks
existing_taking_cols.sort(key=lambda x: tasks.index(x.split('/')[1]) if '/' in x and x.split('/')[1] in tasks else len(tasks))


In [44]:
def show_metrics(history, step, existing_taking_cols=existing_taking_cols):
# get history when trainer/global_step=step
    filtered_history = history[history['trainer/global_step'] == step]
# get filtered history with existing_task_metric_cols
    filtered_history_with_metrics = filtered_history[existing_taking_cols].dropna()

    for key, item in filtered_history_with_metrics.items():
        value = item.values[0]
        value = float(value)
        if "bleu" in key or "rouge" in key or "meteor" in key:
            value /= 100
        if any(
            metric in key for metric in [
                'exact_match_ratio',
                'bleu_selfies',
                'RDK_FTS',
                'MACCS_FTS',
                'morgan_FTS',
                'validity_ratio',
                'roc_auc',
                'bleu2',
                'bleu4',
                'rouge1',
                'rouge2',
                'rougeL',
                'meteor'
                ]
        ):
            # round to 3 decimal places
            value = round(value, 3)
        if "roc_auc" in key:
            value *= 100
        print(f"{key}: {value}")

In [45]:
show_metrics(hist['s_1ep'], step=3229)

val/smol-property_prediction-esol/rmse: 1.3600176027808968
val/smol-property_prediction-lipo/rmse: 0.9537319275503402
val/qm9_homo/mae: 0.004398099415204679
val/qm9_lumo/mae: 0.004335981308411215
val/qm9_homo_lumo_gap/mae: 0.005509077155824507
val/bace/roc_auc: 80.80000000000001
val/smol-property_prediction-bbbp/roc_auc: 84.3
val/smol-property_prediction-clintox/roc_auc: 85.0
val/smol-property_prediction-hiv/roc_auc: 76.5
val/smol-property_prediction-sider/roc_auc: 76.1
val/forward_reaction_prediction/exact_match_ratio: 0.893
val/forward_reaction_prediction/bleu_selfies: 0.963
val/forward_reaction_prediction/RDK_FTS: 0.968
val/forward_reaction_prediction/MACCS_FTS: 0.983
val/forward_reaction_prediction/morgan_FTS: 0.96
val/forward_reaction_prediction/validity_ratio: 1.0
val/smol-forward_synthesis/exact_match_ratio: 0.584
val/smol-forward_synthesis/bleu_selfies: 0.867
val/smol-forward_synthesis/RDK_FTS: 0.847
val/smol-forward_synthesis/MACCS_FTS: 0.904
val/smol-forward_synthesis/morgan_

In [46]:
show_metrics(hist['s+g_1ep'], step=3229)

val/smol-property_prediction-esol/rmse: 1.3558772885536423
val/smol-property_prediction-lipo/rmse: 0.9648915447588865
val/qm9_homo/mae: 0.004164619883040935
val/qm9_lumo/mae: 0.004424143302180685
val/qm9_homo_lumo_gap/mae: 0.0052264750378214824
val/bace/roc_auc: 81.2
val/smol-property_prediction-bbbp/roc_auc: 82.0
val/smol-property_prediction-clintox/roc_auc: 84.5
val/smol-property_prediction-hiv/roc_auc: 78.10000000000001
val/smol-property_prediction-sider/roc_auc: 76.6
val/forward_reaction_prediction/exact_match_ratio: 0.907
val/forward_reaction_prediction/bleu_selfies: 0.97
val/forward_reaction_prediction/RDK_FTS: 0.975
val/forward_reaction_prediction/MACCS_FTS: 0.986
val/forward_reaction_prediction/morgan_FTS: 0.966
val/forward_reaction_prediction/validity_ratio: 1.0
val/smol-forward_synthesis/exact_match_ratio: 0.598
val/smol-forward_synthesis/bleu_selfies: 0.87
val/smol-forward_synthesis/RDK_FTS: 0.854
val/smol-forward_synthesis/MACCS_FTS: 0.908
val/smol-forward_synthesis/morgan_